In [ ]:
from pathlib import Path

from hydra import compose, initialize_config_dir

from samuel.config import RLConfig
import torch

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
config_dir = repo_root / "configs"

with initialize_config_dir(version_base=None, config_dir=str(config_dir)):
    hydra_cfg = compose(
        config_name="rl_train",
        overrides=["run.name=notebook"],  # run.name is a mandatory (???) field
    )

cfg = RLConfig.from_hydra(hydra_cfg)
cfg

In [ ]:
import numpy as np
from IPython.display import display, Audio

from samuel.pink_trombone import SAMPLE_RATE


def display_audio(audio):
    if isinstance(audio, torch.Tensor):
        audio = audio.detach().cpu().numpy()
    audio = np.asarray(audio)
    audio = audio.squeeze()
    if audio.ndim != 1:
        raise ValueError(
            f"Expected a single audio clip, got shape {audio.shape}. "
            "Index into the batch dimension first, e.g. display_audio(x[0])."
        )
    display(Audio(audio, rate=SAMPLE_RATE))


display_audio(batch["audio"][0])

In [ ]:
from samuel.data import build_dataloader


loader = build_dataloader(
    cfg.data,
    batch_size=cfg.batch_size,
    rank=0,
    world_size=1,
    epoch=0,
    seed=cfg.run.seed,
    samples_per_frame=cfg.model.samples_per_frame,
)
data_iter = iter(loader)

In [ ]:
for i in range(1):
    try:
        batch = next(data_iter)
    except StopIteration:
        data_iter = iter(loader)
        batch = next(data_iter)

In [ ]:
batch

In [ ]:
from samuel.model import PinkTromboneController


def _resolve_checkpoint(ref: str) -> Path:
    """Return a local path to the checkpoint file.

    ``ref`` is either a local ``.pt`` path or a wandb artifact reference
    (``entity/project/name:alias``), which is downloaded on demand.
    """
    if Path(ref).exists():
        return Path(ref)

    import wandb

    artifact = wandb.Api().artifact(ref, type="model")
    return Path(artifact.download()) / "last.pt"


def _load_checkpoint(
    model: PinkTromboneController, ref: str, device: torch.device
) -> None:
    ckpt = torch.load(_resolve_checkpoint(ref), map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model"])


model = PinkTromboneController(cfg.model).to(device)
model.eval()

samples_per_frame = cfg.model.samples_per_frame
if cfg.checkpoint is not None:
    _load_checkpoint(model, cfg.checkpoint, device)

In [ ]:
cfg.checkpoint

In [ ]:
from einops import rearrange

from samuel.pink_trombone import pink_trombone_ola

with torch.no_grad():
    params = model(
        rearrange(batch["audio"].to(device), "b s -> b 1 s"),
        batch["pitch"].to(device),
    )
# fixed_seed = torch.arange(start, end, dtype=torch.long)
ola = pink_trombone_ola(
    params,
    seed=0,
    ir_length=cfg.synth.ir_length,
    ir_impl=cfg.synth.ir_impl,
    control_rate=cfg.model.frame_rate,
)

In [ ]:
from samuel.evals.asr import WhisperEvaluator


whisper = WhisperEvaluator(
    model_size=cfg.log.asr_whisper_size,
    device="cuda" if device.type == "cuda" else "cpu",
)

In [ ]:
batch["audio"][0].detach().cpu().numpy().shape

In [ ]:
ola.shape

In [ ]:
display_audio(ola[1])

In [ ]:
display_audio(batch["audio"][1])

In [ ]:
batch["audio"].shape

In [ ]:
import textwrap

import tqdm.auto as tqdm

for i in tqdm.trange(batch["audio"].shape[0]):
    asr_scores = whisper.score(
        batch["audio"][i].detach().cpu().numpy(),
        ola[i].detach().cpu().numpy()[::-1],
        sr=cfg.data.sample_rate,
        target_key=None,
    )
    if asr_scores.hyp != "":
        print("\n".join(textwrap.wrap(str(asr_scores))))

In [ ]:
ola.detach().cpu().numpy().shape

In [ ]:
display_audio(ola[0])